# Raw line family only experiment

Ten notebook jest calkowicie odseparowany od poprzednich eksperymentow.

Obecna wersja robi:
- wczytanie obrazu,
- grayscale,
- odszumienie,
- binaryzacje,
- cleanup,
- przypisanie surowych segmentow do rodzin `horizontal` i `vertical`,
- budowe wstepnych `LogicalLine`,
- render `raw line families` na `cleanup`,
- repair,
- pixel-validated laczenie koncow linii na `repair`,
- budowe prostokatow tolerancji,
- render logical lines i prostokatow tolerancji.

Ta wersja nadal nie liczy:
- warp,
- finalnej siatki planszy,
- komorek,
- OCR cyfr.

In [15]:
import raw_line_family_only_bootstrap as bootstrap
from raw_line_family_only_pipeline import (
    RawLineFamilyArtifacts,
    configure_manual_image_path,
    describe_raw_line_family_artifacts,
    resolve_active_image_selection,
)

In [16]:
import importlib
import raw_line_family_only_pipeline as pipeline

bootstrap = importlib.reload(bootstrap)
pipeline = importlib.reload(pipeline)
API = bootstrap.load_raw_line_family_only_api()

ExperimentConfig = API.ExperimentConfig
REPO_ROOT = API.REPO_ROOT
RawLineFamilyArtifacts = pipeline.RawLineFamilyArtifacts
configure_manual_image_path = pipeline.configure_manual_image_path
describe_raw_line_family_artifacts = pipeline.describe_raw_line_family_artifacts
resolve_active_image_selection = pipeline.resolve_active_image_selection

print("Reloaded raw line family only API, pipeline, and local modules.")

Reloaded raw line family only API, pipeline, and local modules.


In [17]:
CONFIG = ExperimentConfig(
    raw_hough_threshold=35,
    raw_min_line_length_ratio=0.08,
    raw_max_line_gap_ratio=0.005,
)

print(f"Repo root: {REPO_ROOT}")
print(f"Dataset root: {CONFIG.dataset_root}")
print(
    "Pipeline: "
    f"median_{CONFIG.median_kernel_size} -> "
    "gaussian_block"
    f"{CONFIG.adaptive_threshold_block_size}_c"
    f"{CONFIG.adaptive_threshold_c_value} -> "
    "adaptive_plus_components_soft -> directional_close"
)
print(
    "Tolerance rectangles: "
    f"length={CONFIG.tolerance_rectangle_vector_length_px}, "
    f"padding={CONFIG.tolerance_rectangle_padding_px}"
)

Repo root: /home/wojtek/projects/sudoku
Dataset root: /home/wojtek/projects/sudoku/data/raw/boards
Pipeline: median_5 -> gaussian_block11_c2 -> adaptive_plus_components_soft -> directional_close
Tolerance rectangles: length=350, padding=18


## 1. Wybor obrazka

Mozesz wpisac reczna sciezke albo zostawic puste `IMAGE_PATH_INPUT` i wybrac obraz przez `CONFIG.selected_dataset_index`.

In [18]:
IMAGE_PATH_INPUT = "/home/wojtek/projects/sudoku/examples/uploads/image1087.jpg"
# Przyklad:
# IMAGE_PATH_INPUT = "data/raw/boards/przyklad/board_01.jpg"
# IMAGE_PATH_INPUT = ""
3
print(configure_manual_image_path(CONFIG, IMAGE_PATH_INPUT, REPO_ROOT))

Manual image path enabled: /home/wojtek/projects/sudoku/examples/uploads/image1087.jpg


In [19]:
IMAGE_SELECTION = resolve_active_image_selection(CONFIG, API)
ACTIVE_IMAGE_PATH = IMAGE_SELECTION.active_image_path

for line in IMAGE_SELECTION.preview_lines:
    print(line)

Found 250 image(s) under dataset root.
[00] mixed/image1.jpg
[01] mixed/image10.jpg
[02] mixed/image100.jpg
[03] mixed/image102.jpg
[04] mixed/image103.jpg
[05] mixed/image104.jpg
[06] mixed/image106.jpg
[07] mixed/image107.jpg
[08] mixed/image109.jpg
[09] mixed/image111.jpg
[10] mixed/image112.jpg
[11] mixed/image113.jpg
[12] mixed/image114.jpg
[13] mixed/image117.jpg
[14] mixed/image118.jpg
[15] mixed/image119.jpg
[16] mixed/image120.jpg
[17] mixed/image122.jpg
[18] mixed/image125.jpg
[19] mixed/image126.jpg
... and 230 more

Active image: /home/wojtek/projects/sudoku/examples/uploads/image1087.jpg


In [20]:
import cv2

SOURCE_BGR = API.load_image_bgr(ACTIVE_IMAGE_PATH)
DISPLAY_BGR = API.resize_for_display(SOURCE_BGR, CONFIG.max_display_size)
GRAY_IMAGE = cv2.cvtColor(DISPLAY_BGR, cv2.COLOR_BGR2GRAY)

DENOISE_NAME = f"median_{CONFIG.median_kernel_size}"
DENOISED_IMAGE = API.apply_median_denoise(
    GRAY_IMAGE,
    CONFIG,
)

THRESHOLD_NAME = (
    "gaussian_block"
    f"{CONFIG.adaptive_threshold_block_size}_c{CONFIG.adaptive_threshold_c_value}"
)
BINARY_IMAGE = API.apply_gaussian_threshold(
    DENOISED_IMAGE,
    CONFIG,
)

CLEANUP_NAME = "adaptive_plus_components_soft"
MIN_COMPONENT_AREA_PX, CLEAN_BINARY = API.apply_soft_component_cleanup(
    BINARY_IMAGE,
    CONFIG,
)

FAMILY_DETECTION_RESULT = API.detect_line_families(
    CLEAN_BINARY,
    CONFIG,
    include_logical_lines=False,
)
CLEANUP_BINARY_FAMILY_OVERLAY, SOURCE_FAMILY_OVERLAY = API.build_line_family_overlays(
    DISPLAY_BGR,
    CLEAN_BINARY,
    FAMILY_DETECTION_RESULT,
    CONFIG,
)


In [21]:
import time

REPAIR_NAME = "directional_close"
repair_started_at = time.perf_counter()
REPAIRED_BINARY = API.apply_directional_close_repair(
    CLEAN_BINARY,
    CONFIG,
)
repair_elapsed_s = time.perf_counter() - repair_started_at

PIXEL_CONNECTION_BINARY_NAME = "repair"
# Do testu mozna przelaczyc na:
# PIXEL_CONNECTION_BINARY_NAME = "cleanup"

if PIXEL_CONNECTION_BINARY_NAME == "cleanup":
    PIXEL_CONNECTION_BINARY = CLEAN_BINARY
elif PIXEL_CONNECTION_BINARY_NAME == "repair":
    PIXEL_CONNECTION_BINARY = REPAIRED_BINARY
else:
    raise ValueError(
        "PIXEL_CONNECTION_BINARY_NAME must be 'cleanup' or 'repair'."
    )

full_detection_started_at = time.perf_counter()
FINAL_LINE_FAMILY_RESULT = API.detect_line_families(
    CLEAN_BINARY,
    CONFIG,
    pixel_connection_binary_image=PIXEL_CONNECTION_BINARY,
)
full_detection_elapsed_s = time.perf_counter() - full_detection_started_at

PIXEL_RENDER_BINARY_NAME = (
    f"{PIXEL_CONNECTION_BINARY_NAME} binary used for pixel connection"
)
BINARY_LOGICAL_LINE_OVERLAY, SOURCE_LOGICAL_LINE_OVERLAY = API.build_logical_line_overlays(
    DISPLAY_BGR,
    PIXEL_CONNECTION_BINARY,
    FINAL_LINE_FAMILY_RESULT,
    CONFIG,
)
(
    BINARY_LOGICAL_LINE_INTERSECTION_OVERLAY,
    SOURCE_LOGICAL_LINE_INTERSECTION_OVERLAY,
) = API.build_logical_line_intersection_overlays(
    DISPLAY_BGR,
    PIXEL_CONNECTION_BINARY,
    FINAL_LINE_FAMILY_RESULT,
    CONFIG,
)
(
    BINARY_FRAME_OVERLAY,
    SOURCE_FRAME_OVERLAY,
) = API.build_frame_overlays(
    DISPLAY_BGR,
    PIXEL_CONNECTION_BINARY,
    FINAL_LINE_FAMILY_RESULT,
    CONFIG,
)
(
    BINARY_TOLERANCE_RECTANGLE_OVERLAY,
    SOURCE_TOLERANCE_RECTANGLE_OVERLAY,
) = API.build_tolerance_rectangle_overlays(
    DISPLAY_BGR,
    PIXEL_CONNECTION_BINARY,
    FINAL_LINE_FAMILY_RESULT,
    CONFIG,
)

ARTIFACTS = RawLineFamilyArtifacts(
    source_bgr=SOURCE_BGR,
    display_bgr=DISPLAY_BGR,
    gray_image=GRAY_IMAGE,
    denoise_name=DENOISE_NAME,
    denoised_image=DENOISED_IMAGE,
    threshold_name=THRESHOLD_NAME,
    binary_image=BINARY_IMAGE,
    min_component_area_px=MIN_COMPONENT_AREA_PX,
    cleanup_name=CLEANUP_NAME,
    clean_binary=CLEAN_BINARY,
    repair_name=REPAIR_NAME,
    repaired_binary=REPAIRED_BINARY,
    line_family_result=FINAL_LINE_FAMILY_RESULT,
    binary_family_overlay=CLEANUP_BINARY_FAMILY_OVERLAY,
    source_family_overlay=SOURCE_FAMILY_OVERLAY,
    binary_logical_line_overlay=BINARY_LOGICAL_LINE_OVERLAY,
    source_logical_line_overlay=SOURCE_LOGICAL_LINE_OVERLAY,
    binary_logical_line_intersection_overlay=(
        BINARY_LOGICAL_LINE_INTERSECTION_OVERLAY
    ),
    source_logical_line_intersection_overlay=(
        SOURCE_LOGICAL_LINE_INTERSECTION_OVERLAY
    ),
    binary_frame_overlay=BINARY_FRAME_OVERLAY,
    source_frame_overlay=SOURCE_FRAME_OVERLAY,
    binary_tolerance_rectangle_overlay=BINARY_TOLERANCE_RECTANGLE_OVERLAY,
    source_tolerance_rectangle_overlay=SOURCE_TOLERANCE_RECTANGLE_OVERLAY,
)

print(f"Repair time: {repair_elapsed_s:.3f}s")
print(
    "Detect + logical lines + pixel connection time "
    f"with Hough only on cleanup and pixels on {PIXEL_CONNECTION_BINARY_NAME}: "
    f"{full_detection_elapsed_s:.3f}s"
)
print()

for line in describe_raw_line_family_artifacts(ARTIFACTS):
    print(line)

API.plot_named_images(
    [
        (f"cleanup: {CLEANUP_NAME}", CLEAN_BINARY, False),
        ("raw line families on cleanup", CLEANUP_BINARY_FAMILY_OVERLAY, True),
        ("raw line families on source", SOURCE_FAMILY_OVERLAY, True),
        (f"repair: {REPAIR_NAME}", REPAIRED_BINARY, False),
        (PIXEL_RENDER_BINARY_NAME, PIXEL_CONNECTION_BINARY, False),
        (
            f"logical lines on {PIXEL_CONNECTION_BINARY_NAME} binary",
            BINARY_LOGICAL_LINE_OVERLAY,
            True,
        ),
        ("logical lines on source", SOURCE_LOGICAL_LINE_OVERLAY, True),
        (
            f"logical line intersections on {PIXEL_CONNECTION_BINARY_NAME} binary",
            BINARY_LOGICAL_LINE_INTERSECTION_OVERLAY,
            True,
        ),
        (
            "logical line intersections on source",
            SOURCE_LOGICAL_LINE_INTERSECTION_OVERLAY,
            True,
        ),
        (f"frames on {PIXEL_CONNECTION_BINARY_NAME} binary", BINARY_FRAME_OVERLAY, True),
        ("frames on source", SOURCE_FRAME_OVERLAY, True),
        (
            f"tolerance rectangles on {PIXEL_CONNECTION_BINARY_NAME} binary",
            BINARY_TOLERANCE_RECTANGLE_OVERLAY,
            True,
        ),
        ("tolerance rectangles on source", SOURCE_TOLERANCE_RECTANGLE_OVERLAY, True),
    ],
    columns=3,
    figure_scale=5.5,
)

ValueError: horizontal_line must belong to the horizontal family.